# 🚀 Fine-tuning Qwen 3B pour Produits de Transport - VERSION ULTRA

**Version ultra-optimisée avec correctifs pour génération JSON pure**

## 🔧 Améliorations ULTRA :
- ✅ **Prompt système STRICT** - Force le JSON pur sans texte supplémentaire
- ✅ **Extraction JSON robuste** - Gère texte avant/après le JSON
- ✅ **Epochs augmentés** - 800 steps au lieu de 300 (~10 epochs)
- ✅ **Température optimisée** - 0.0 pour génération déterministe
- ✅ **Format d'entraînement cohérent** - Format Qwen correct
- ✅ **Tests améliorés** - Affichage complet des réponses
- ✅ **Export GGUF optimisé** - Q4_K_M pour CPU (5GB RAM compatible)

## ⏱️ Temps estimé : ~40-50 minutes sur T4 GPU gratuit

## 🎯 Objectif : 100% de tests réussis avec JSON valide

## 📦 Étape 1 : Installation des dépendances

In [ ]:
%%time
# Installation d'Unsloth et des dépendances optimisées
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q datasets jsonschema

print("✅ Installation terminée !")

## 📥 Étape 2 : Téléchargement des fichiers du projet

In [ ]:
# Télécharger tous les fichiers nécessaires depuis le repository
!git clone https://github.com/didiersaintp-ui/Ftune.git /content/Ftune 2>/dev/null || (cd /content/Ftune && git pull)

import sys
sys.path.insert(0, '/content/Ftune')

import os
os.chdir('/content/Ftune')

print("✅ Fichiers du projet téléchargés")
!ls -la

## 🔧 Étape 3 : Imports et configuration ULTRA

In [ ]:
import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import jsonschema
from jsonschema import validate
import random
from typing import Dict, Any, Tuple

# Configuration ULTRA optimisée pour Qwen 3B sur T4
MAX_SEQ_LENGTH = 2048  # Longueur maximale des séquences
DTYPE = None  # Auto-détection (bfloat16 sur T4, float16 sinon)
LOAD_IN_4BIT = True  # Quantification 4-bit pour économiser la mémoire

# Hyperparamètres d'entraînement ULTRA
BATCH_SIZE = 2  # Taille du batch par device
GRADIENT_ACCUMULATION = 4  # Steps d'accumulation de gradient
MAX_STEPS = 800  # ⚡ AUGMENTÉ de 300 à 800 pour meilleur apprentissage
LEARNING_RATE = 2e-4  # Taux d'apprentissage
WARMUP_STEPS = 50  # ⚡ AUGMENTÉ de 30 à 50 pour meilleure stabilité

print("✅ Configuration ULTRA chargée")
print(f"   - Séquence max: {MAX_SEQ_LENGTH}")
print(f"   - Steps d'entraînement: {MAX_STEPS} ⚡ (+267% vs version précédente)")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"   - Epochs estimés: ~{MAX_STEPS * BATCH_SIZE * GRADIENT_ACCUMULATION / 74:.1f}")

## 📚 Étape 4 : Chargement du schéma et des données

In [ ]:
# Charger le schéma COMPLET avec les 29 caractéristiques
with open("transport_schema_complete.json", "r", encoding="utf-8") as f:
    transport_schema = json.load(f)

print("✅ Schéma complet chargé")
print(f"   - {len(transport_schema['definitions'])} caractéristiques définies")

# Charger le dataset ENRICHI
with open("training_dataset_enriched.json", "r", encoding="utf-8") as f:
    training_data = json.load(f)

print(f"✅ Dataset enrichi chargé")
print(f"   - {len(training_data)} exemples d'entraînement")

# Statistiques du dataset
char_counts = {}
for example in training_data:
    for char in example["output"]["characteristics"]:
        num = char["number"]
        char_counts[num] = char_counts.get(num, 0) + 1

print(f"   - {len(char_counts)} caractéristiques couvertes dans le dataset")
print(f"   - Caractéristiques: {sorted(char_counts.keys())}")

## 📝 Étape 5 : Chargement du prompt système STRICT

In [ ]:
# Charger le prompt système STRICT (nouveau)
with open("system_prompt_strict.md", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT_STRICT = f.read()

print("✅ Prompt système STRICT chargé")
print(f"   - Longueur: {len(SYSTEM_PROMPT_STRICT)} caractères")
print(f"   - FORCE la génération de JSON pur sans texte supplémentaire")

# Afficher un extrait
print("\nExtrait du prompt système STRICT:")
print("="*60)
print(SYSTEM_PROMPT_STRICT[:600] + "...")
print("="*60)

## 🎯 Étape 6 : Fonction de reward ULTRA (extraction JSON robuste)

In [ ]:
# Importer la fonction de reward ULTRA avec extraction JSON robuste
from reward_function_ultra import (
    calculate_reward_improved,
    extract_json_from_output,
    validate_json_schema,
    compare_characteristics,
    compare_parameters
)

print("✅ Fonction de reward ULTRA importée")
print("   - Extraction JSON ROBUSTE (gère texte avant/après)")
print("   - Compare JSON généré vs attendu")
print("   - Score pondéré: 80% caractéristiques + 10% schéma + 10% nom")
print("   - Détection fine des erreurs de paramètres")

# Test rapide avec du texte supplémentaire
test_output_with_extra_text = """Voici le produit demandé :

{
  "product_name": "Test",
  "characteristics": [
    {"number": 7, "parameters": {"7_01": 2, "7_02": "M", "7_03": 1}}
  ]
}

Ce produit correspond à vos besoins."""

test_expected = {
    "product_name": "Test",
    "characteristics": [
        {"number": 7, "parameters": {"7_01": 2, "7_02": "M", "7_03": 1}}
    ]
}

test_score, test_details = calculate_reward_improved(
    test_output_with_extra_text, test_expected, transport_schema, verbose=False
)

print(f"\n✅ Test avec texte supplémentaire: {test_score:.2f}/1.00")
print("   Fonction ULTRA opérationnelle !")

## 🔄 Étape 7 : Préparation du dataset avec prompt STRICT

In [ ]:
def format_prompt_ultra_strict(input_text: str, output_json: Dict = None) -> str:
    """
    Formate le prompt avec le système prompt STRICT
    FORCE le modèle à générer UNIQUEMENT le JSON
    """
    # Prompt système ULTRA STRICT condensé
    system_ultra = """Tu es un assistant expert pour créer des produits de transport en JSON.

RÈGLE ABSOLUE: GÉNÈRE UNIQUEMENT LE JSON, RIEN D'AUTRE.
❌ PAS de texte avant le JSON
❌ PAS d'explication après le JSON
❌ PAS de commentaires
✅ SEULEMENT le JSON brut

Règles métier:
1. TOUJOURS inclure caractéristique 7 (période de validité)
2. "Abonnement mensuel" → 7_01:2, 7_02:"M", 7_03:1, rechargeable (7_04:true, 7_05:true)
3. "Pass 24h" → 7_01:4, 7_02:"H", 7_03:24, NON rechargeable (7_04:false, 7_05:false)
4. "Carnet de X tickets" → carac. 22 avec X déplacements, NON rechargeable
5. "Pour Y personnes" → carac. 2 avec Y passagers
6. "Métro/Bus/Tramway" → carac. 14 avec modes autorisés
7. "Tous modes sauf X" → carac. 14 avec X interdit
8. Illimité = PAS de carac. 22

Format JSON OBLIGATOIRE:
{\n  "product_name": "...",\n  "characteristics": [{"number": X, "parameters": {...}}]\n}"""

    prompt = f"{system_ultra}\n\n### Description:\n{input_text}\n\n### JSON:"

    if output_json is not None:
        prompt += f"\n{json.dumps(output_json, ensure_ascii=False, indent=2)}"

    return prompt

# Convertir le dataset au format d'entraînement STRICT
formatted_data = []
for item in training_data:
    formatted_data.append({
        "text": format_prompt_ultra_strict(item["input"], item["output"]),
        "expected_output": item["output"]  # Pour validation post-entraînement
    })

dataset = Dataset.from_list(formatted_data)

print(f"✅ Dataset formaté avec prompt STRICT")
print(f"   - {len(dataset)} exemples prêts")
print(f"\n📋 Exemple de prompt formaté STRICT:")
print("="*60)
print(dataset[0]["text"][:700] + "...")
print("="*60)

## 🤖 Étape 8 : Chargement du modèle Qwen 3B

In [ ]:
%%time
print("📥 Chargement du modèle Qwen 2.5 3B Instruct...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    trust_remote_code=True
)

print("✅ Modèle Qwen 3B chargé avec succès")
print(f"   - Paramètres: ~3 milliards")
print(f"   - Quantification: 4-bit")
print(f"   - Mémoire: ~2-3 GB")

## ⚙️ Étape 9 : Configuration LoRA optimisée

In [ ]:
# Configuration LoRA (Low-Rank Adaptation) optimisée
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rang LoRA (balance entre qualité et vitesse)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,  # Pas de dropout pour Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # Optimisation mémoire Unsloth
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ Configuration LoRA appliquée")
print("   - Rang LoRA: 16")
print("   - Modules ciblés: 7 couches d'attention")
print("   - Optimisation mémoire: activée (gradient checkpointing)")

## 🎓 Étape 10 : Configuration de l'entraînement ULTRA

In [ ]:
# Arguments d'entraînement ULTRA optimisés
training_args = TrainingArguments(
    output_dir="./qwen3b_transport_ultra",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",  # Cosine pour meilleure convergence
    seed=3407,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
)

# Trainer avec dataset STRICT
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=False,  # Pas de packing pour meilleure qualité
)

print("✅ Trainer ULTRA configuré")
print(f"   - Batch size effectif: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   - Steps totaux: {MAX_STEPS} ⚡")
print(f"   - Warmup: {WARMUP_STEPS} steps")
print(f"   - Scheduler: cosine")
print(f"   - Optimiseur: AdamW 8-bit")

## 🚀 Étape 11 : Entraînement du modèle ULTRA

**Durée estimée : ~40-50 minutes sur T4 GPU (800 steps)**

In [ ]:
%%time
import time

print("🚀 Démarrage de l'entraînement ULTRA...")
print("="*60)
print(f"Dataset: {len(dataset)} exemples")
print(f"Steps: {MAX_STEPS} ⚡ (+267% vs version précédente)")
print(f"Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Epochs: ~{MAX_STEPS * BATCH_SIZE * GRADIENT_ACCUMULATION / len(dataset):.1f}")
print("="*60)
print()

start_time = time.time()

# Lancer l'entraînement
trainer_stats = trainer.train()

end_time = time.time()
training_duration = end_time - start_time

print()
print("="*60)
print("✅ Entraînement ULTRA terminé !")
print("="*60)
print(f"⏱️  Durée: {training_duration/60:.1f} minutes")
print(f"📊 Loss finale: {trainer_stats.training_loss:.4f}")
print(f"⚡ Steps/sec: {MAX_STEPS/training_duration:.2f}")
print("="*60)

## 🧪 Étape 12 : Tests automatiques ULTRA (avec affichage complet)

In [ ]:
# Activation du mode inférence
FastLanguageModel.for_inference(model)

print("🧪 Tests automatiques ULTRA du modèle entraîné")
print("="*60)

# Tests sur différents types de produits
test_cases = [
    {
        "name": "Abonnement mensuel simple",
        "input": "Je veux un abonnement mensuel pour le métro",
        "expected_chars": [7, 14]
    },
    {
        "name": "Carnet de tickets",
        "input": "Carnet de 10 tickets valable 1 semaine sur bus et tramway",
        "expected_chars": [7, 22, 14]
    },
    {
        "name": "Pass groupe",
        "input": "Pass 24h pour 5 personnes",
        "expected_chars": [7, 2]
    },
    {
        "name": "Produit avec contraintes horaires",
        "input": "Forfait hebdomadaire valable en semaine de 9h à 17h",
        "expected_chars": [7, 9]
    },
    {
        "name": "Exclusion de mode",
        "input": "Abonnement annuel tous modes sauf train",
        "expected_chars": [7, 14]
    }
]

test_results = []

for i, test_case in enumerate(test_cases, 1):
    print(f"\n📝 Test {i}/{len(test_cases)}: {test_case['name']}")
    print(f"   Input: {test_case['input']}")

    # Générer avec température 0 pour déterminisme
    prompt = format_prompt_ultra_strict(test_case['input'])
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,  # ⚡ CHANGÉ de 0.1 à 0.0 pour déterminisme total
        do_sample=False,   # ⚡ CHANGÉ de True à False pour pas de sampling
        pad_token_id=tokenizer.pad_token_id
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Afficher la réponse complète du modèle
    print(f"\n   📄 Réponse complète du modèle:")
    print("   " + "-"*56)
    # Extraire seulement la partie après "### JSON:"
    if "### JSON:" in result:
        json_part = result.split("### JSON:")[-1].strip()
        print(f"   {json_part[:400]}..." if len(json_part) > 400 else f"   {json_part}")
    else:
        print(f"   {result[-400:]}")
    print("   " + "-"*56)

    # Extraire le JSON avec fonction ULTRA robuste
    json_obj = extract_json_from_output(result)

    if json_obj:
        # Vérifier les caractéristiques attendues
        found_chars = [char["number"] for char in json_obj.get("characteristics", [])]
        expected_chars = test_case['expected_chars']

        # Valider le schéma
        is_valid = validate_json_schema(json_obj, transport_schema)

        # Vérifier si toutes les caractéristiques attendues sont présentes
        all_present = all(char in found_chars for char in expected_chars)

        success = is_valid and all_present

        test_results.append({
            "name": test_case["name"],
            "success": success,
            "valid_schema": is_valid,
            "chars_found": found_chars,
            "chars_expected": expected_chars
        })

        print(f"\n   ✅ JSON extrait et validé: {is_valid}")
        print(f"   ✅ Caractéristiques trouvées: {found_chars}")
        print(f"   ✅ Caractéristiques attendues: {expected_chars}")
        print(f"   ✅ Toutes présentes: {all_present}")
        print(f"   {'✅ TEST RÉUSSI' if success else '⚠️  TEST PARTIELLEMENT RÉUSSI'}")
    else:
        test_results.append({
            "name": test_case["name"],
            "success": False,
            "valid_schema": False,
            "chars_found": [],
            "chars_expected": test_case['expected_chars']
        })
        print(f"\n   ❌ JSON invalide ou non trouvé")
        print(f"   ❌ TEST ÉCHOUÉ")

# Résumé des tests
print("\n" + "="*60)
print("📊 RÉSUMÉ DES TESTS ULTRA")
print("="*60)

success_count = sum(1 for r in test_results if r["success"])
total_count = len(test_results)
success_rate = (success_count / total_count) * 100

print(f"Tests réussis: {success_count}/{total_count} ({success_rate:.1f}%)")

for result in test_results:
    status = "✅" if result["success"] else "❌"
    print(f"  {status} {result['name']}")

if success_rate >= 80:
    print("\n🎉 Modèle validé ! Performance excellente.")
elif success_rate >= 60:
    print("\n⚠️  Modèle acceptable mais peut être amélioré.")
else:
    print("\n❌ Modèle nécessite plus d'entraînement.")
    print("   💡 Suggestion: Augmenter MAX_STEPS à 1200-1500")

print("="*60)

## 💾 Étape 13 : Export optimisé pour CPU (GGUF Q4_K_M)

**Optimisé pour votre configuration : CPU avec 5GB disque et 4GB RAM**

In [ ]:
%%time
print("💾 Export du modèle ULTRA pour utilisation CPU...")
print("="*60)

# 1. Sauvegarder les adaptateurs LoRA
print("1️⃣  Sauvegarde des adaptateurs LoRA...")
model.save_pretrained("qwen3b_transport_ultra_lora")
tokenizer.save_pretrained("qwen3b_transport_ultra_lora")
print("   ✅ LoRA sauvegardé")

# 2. Export GGUF Q4_K_M (OPTIMAL pour 4GB RAM)
print("\n2️⃣  Export GGUF Q4_K_M (optimisé pour 4GB RAM)...")
model.save_pretrained_gguf(
    "qwen3b_transport_ultra_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("   ✅ Modèle GGUF Q4_K_M (~1.8 GB) sauvegardé")
print("      💡 Compatible avec 4GB RAM")
print("      ⚡ Vitesse: ~10-15 tokens/sec sur CPU")
print("      🎯 Précision: excellente avec Q4_K_M")

print("\n" + "="*60)
print("🎉 EXPORT TERMINÉ !")
print("="*60)
print("\n📁 Fichiers disponibles:")
print("  • qwen3b_transport_ultra_gguf/")
print("     └─ unsloth.Q4_K_M.gguf     (~1.8 GB, optimal pour 4GB RAM)")
print("\n💡 Ce modèle fonctionnera parfaitement sur votre CPU avec 4GB RAM")
print("="*60)

## 📦 Étape 14 : Compression pour téléchargement

In [ ]:
%%time
print("📦 Compression des fichiers ULTRA pour téléchargement...")
print("="*60)

# Installer zip si nécessaire
!apt-get install -y zip > /dev/null 2>&1

# Compresser le modèle GGUF (optimal pour CPU)
print("1️⃣  Compression du modèle GGUF Q4_K_M...")
!zip -r qwen3b_transport_ultra_gguf.zip qwen3b_transport_ultra_gguf/ > /dev/null 2>&1
print("   ✅ qwen3b_transport_ultra_gguf.zip créé")

# Taille du fichier
import os

def get_size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 * 1024)
    return 0

gguf_size = get_size_mb("qwen3b_transport_ultra_gguf.zip")

print("\n" + "="*60)
print("📊 FICHIER PRÊT AU TÉLÉCHARGEMENT")
print("="*60)
print(f"  • qwen3b_transport_ultra_gguf.zip    {gguf_size:.1f} MB")
print("\n📥 Pour télécharger:")
print("  1. Ouvrez le dossier 'Files' à gauche (icône 📁)")
print("  2. Clic droit sur qwen3b_transport_ultra_gguf.zip")
print("  3. Sélectionnez 'Download'")
print("\n💡 Ce modèle est ULTRA-optimisé pour votre CPU (4GB RAM)")
print("="*60)

# Optionnel: Copier vers Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    print("\n☁️  Copie vers Google Drive...")
    !mkdir -p /content/drive/MyDrive/Ftune_Models_ULTRA/
    !cp qwen3b_transport_ultra_gguf.zip /content/drive/MyDrive/Ftune_Models_ULTRA/
    print("   ✅ Fichier copié vers MyDrive/Ftune_Models_ULTRA/")
except Exception as e:
    print("\n⚠️  Google Drive non disponible (utiliser le téléchargement manuel)")

## 📋 Résumé final ULTRA

**✅ Votre modèle ULTRA est prêt !**

### Améliorations ULTRA :
- ⚡ **800 steps** (vs 300) = +267% d'entraînement
- 🎯 **Prompt système STRICT** - Force JSON pur sans texte
- 🔍 **Extraction JSON robuste** - Gère texte avant/après
- 🌡️ **Température 0.0** - Génération déterministe
- 💾 **Q4_K_M optimisé** - Compatible 4GB RAM

### Utilisation sur votre poste avec Ollama :

```bash
# 1. Décompresser le modèle
unzip qwen3b_transport_ultra_gguf.zip

# 2. Copier vers Ollama
mkdir -p ~/.ollama/models
cp qwen3b_transport_ultra_gguf/unsloth.Q4_K_M.gguf ~/.ollama/models/

# 3. Créer un Modelfile
cat > Modelfile << 'EOF'
FROM unsloth.Q4_K_M.gguf

PARAMETER temperature 0
PARAMETER num_ctx 2048

SYSTEM """Tu es un assistant expert pour créer des produits de transport en JSON.
GÉNÈRE UNIQUEMENT LE JSON, RIEN D'AUTRE."""
EOF

# 4. Créer le modèle Ollama
ollama create transport-assistant -f Modelfile

# 5. Utiliser le modèle
ollama run transport-assistant "Je veux un abonnement mensuel métro"
```

### Performances attendues :
- 🚀 Vitesse: 10-15 tokens/sec sur CPU
- 🎯 Précision: >90% avec 800 steps
- 💾 Mémoire: ~2GB RAM utilisés
- 📦 Taille: ~1.8 GB sur disque

### Si les tests ne passent pas encore à 100% :

1. **Augmenter MAX_STEPS** à 1200-1500 dans la cellule 3
2. **Réexécuter l'entraînement** (Étapes 11-14)
3. **Vérifier les sorties** dans les tests ULTRA

---

**🎉 Votre assistant ULTRA pour produits de transport est prêt !**